# Embedding Fine-tuning with NeMo Microservices

Fine-tune an embedding model and improve retrieval by 6-10% in ~1 hour.

## Prerequisites

1. **Deploy NeMo Microservices 25.8.0+:**  
   https://docs.nvidia.com/nemo/microservices/latest/get-started/setup/minikube/

2. **Register base model:**
   ```bash
   helm upgrade nemo nmp/nemo-microservices-helm-chart --reuse-values \
     --set 'customizer.customizationTargets.targets.nvidia/llama-3\.2-nv-embedqa-1b@v2.enabled=true'
   kubectl delete pod -l app.kubernetes.io/name=nemo-customizer
   ```
   *Note: Pod will automatically redeploy and target will become available*

3. **Get HuggingFace token:** https://huggingface.co/settings/tokens

Then update the config below and run all cells.

## Overview

This notebook demonstrates the complete workflow for fine-tuning an embedding model using NeMo Microservices (NMP). You'll take a base embedding model, adapt it to scientific domain data, deploy it as a production NVIDIA Inference Microservice (NIM), and measure the performance improvement on a benchmark retrieval task.


## Objectives

By the end of this notebook, you will:
- Fine-tune [`nvidia/llama-3.2-nv-embedqa-1b-v2`](https://build.nvidia.com/nvidia/llama-3_2-nv-embedqa-1b-v2) on 65K scientific paper triplets from [SPECTER dataset](https://huggingface.co/datasets/embedding-data/SPECTER)
- Deploy the fine-tuned model as a production-ready NIM inference service
- Evaluate retrieval performance on the [SciDocs benchmark](https://huggingface.co/datasets/BeIR/scidocs)
- Achieve measurable improvement: baseline recall@5 of 0.159 → ~0.176 (+6-10% improvement)

*Note: To recreate baseline (0.159), deploy the base `nvidia/llama-3.2-nv-embedqa-1b-v2` model (Step 4) and run evaluation (Step 6).*

In [ ]:
# Install required packages
%pip install -q datasets huggingface_hub openai nemo-microservices

In [ ]:
# Endpoints and configuration
NDS_URL = "http://data-store.test"     # Your Data Store URL
NEMO_URL = "http://nemo.test"          # Your NeMo Microservices URL
NIM_URL = "http://nim.test"            # Your NIM URL
HF_TOKEN = ""                          # Your HuggingFace token
NS = ""                        # # Use unique namespace e.g. yourname_ns

if not HF_TOKEN or not NS:
    raise Exception("Please make sure your HF_TOKEN and NS are added above")

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore', category=Warning, module='tqdm')

from datasets import load_dataset
from nemo_microservices import NeMoMicroservices
from huggingface_hub import HfApi
from openai import OpenAI
import json, os, requests
from time import sleep, time

In [ ]:
# Initialize NeMo client
nemo = NeMoMicroservices(base_url=NEMO_URL, inference_base_url=NIM_URL)
print("NeMo client initialized")

## Step 1: Prepare Data

Download 10% of the SPECTER dataset containing ~684K scientific paper triplets (query, positive, negative) and format for embedding fine-tuning.

**Dataset format:** Each triplet teaches the model via contrastive learning to maximize similarity between query and positive document while minimizing similarity between query and negative document.


In [ ]:
# 1. Prepare data (~3 min)
# Using 10% of dataset (68K of 684K triplets) for faster training - scale to full dataset for better results
print("Downloading SPECTER dataset...")
os.environ["HF_TOKEN"] = HF_TOKEN
data = load_dataset("embedding-data/SPECTER")['train'].shuffle(seed=42).select(range(68400))

print("Splitting dataset...")
splits = data.train_test_split(test_size=0.10, seed=42)
val = splits['test'].train_test_split(test_size=0.50, seed=42)['train']

print("Saving to JSONL...")
os.makedirs("data", exist_ok=True)
for name, d in [("train", splits['train']), ("val", val)]:
    with open(f"data/{name}.jsonl", "w") as f:
        for row in d:
            f.write(json.dumps({"query": row['set'][0], "pos_doc": row['set'][1], "neg_doc": [row['set'][2]]}) + "\n")

print(f"Data prepared: {len(splits['train'])} train, {len(val)} val")

# Example triplet
example = splits['train'][0]
print(f"\nExample triplet:")
print(f"Query:    {example['set'][0]}")
print(f"Positive: {example['set'][1]}")
print(f"Negative: {example['set'][2]}")


## Step 2: Upload to NeMo

Upload the prepared data to NeMo Data Store and register it for training.


In [ ]:
# 2. Upload data to NeMo Data Store (~5 min)
# Data Store: Storage for training/eval datasets https://docs.nvidia.com/nim-operator/latest/data-store.html
# Entity Store: Registry for models, configs, metadata https://docs.nvidia.com/nim-operator/latest/entity-store.html
print("Creating namespace...")
nemo.namespaces.create(id=NS)
requests.post(f"{NDS_URL}/v1/datastore/namespaces", data={"namespace": NS})

print("Creating data repository...")
hf = HfApi(endpoint=f"{NDS_URL}/v1/hf", token=None)
hf.create_repo(f"{NS}/data", repo_type='dataset')

print("Uploading files...")
hf.upload_file(path_or_fileobj="data/train.jsonl", path_in_repo="training/training.jsonl", repo_id=f"{NS}/data", repo_type='dataset')
hf.upload_file(path_or_fileobj="data/val.jsonl", path_in_repo="validation/validation.jsonl", repo_id=f"{NS}/data", repo_type='dataset')

print("Registering dataset...")
nemo.datasets.create(name="data", namespace=NS, files_url=f"hf://datasets/{NS}/data")

print("Data uploaded")


## Step 3: Train Model

Fine-tune the embedding model using supervised contrastive learning (~30 minutes).


In [ ]:
# 3. Train model (~30 min)
print("Creating training config...")
nemo.customization.configs.create(
    name="cfg@v1", 
    namespace=NS, 
    target="nvidia/llama-3.2-nv-embedqa-1b@v2", 
    training_options=[{"training_type": "sft", "finetuning_type": "all_weights", "num_gpus": 1, "micro_batch_size": 8}], 
    max_seq_length=2048)

In [ ]:
# Customizer: Orchestrates model training on GPUs https://docs.nvidia.com/nim-operator/latest/customizer.html
print("Starting training job...")
job = nemo.customization.jobs.create(
    name="job", 
    config=f"{NS}/cfg@v1", 
    dataset={"namespace": NS, "name": "data"},
    hyperparameters={"finetuning_type": "all_weights", "epochs": 1, "batch_size": 256, "learning_rate": 5e-6}, 
    output_model=f"{NS}/model")

print(f"Training job: {job.id}")
print("Training takes ~30 minutes...\n")

In [ ]:
# Training progress
last_step = -1
while True:
    job_status = nemo.customization.jobs.retrieve(job.id)
    if job_status.status not in ["pending", "created", "running"]:
        break
    
    d = job_status.status_details
    elapsed_min = int(d.elapsed_time) // 60
    elapsed_sec = int(d.elapsed_time) % 60
    
    if d.epochs_completed >= 1:
        print(f"\rTraining complete: saving model... | {elapsed_min}m {elapsed_sec}s", end="")
    elif d.metrics and d.metrics.metrics.train_loss:
        step = d.metrics.metrics.train_loss[-1].step
        if step != last_step:
            if last_step == -1:
                print()
            loss = d.metrics.metrics.train_loss[-1].value
            pct = int(step / d.steps_per_epoch * 100) if d.steps_per_epoch else 0
            print(f"{pct}% | Step {step} | Loss: {loss:.4f} | {elapsed_min}m {elapsed_sec}s")
            last_step = step
    else:
        print(f"\rInitializing: loading model and data... | {elapsed_min}m {elapsed_sec}s", end="")
    
    sleep(10)

print(f"\n\nTraining complete | Time: {int(job_status.status_details.elapsed_time)//60}m")

## Step 4: Deploy Model

Deploy the fine-tuned model as a NIM inference service (~5 minutes).


In [ ]:
# 4. Deploy model (~5 min)
print("Deploying model...")

try:
    existing = nemo.deployment.model_deployments.retrieve(deployment_name="nim", namespace=NS)
    print(f"Deployment 'nim' already exists (status: {existing.status_details.status})")
except:
    nemo.deployment.model_deployments.create(
        name="nim",
        namespace=NS,
        config={
            "model": f"{NS}/model@{job.id}",
            "nim_deployment": {
                "image_name": "nvcr.io/nim/nvidia/llama-3.2-nv-embedqa-1b-v2",
                "image_tag": "1.6.0",
                "gpu": 1,
                "disable_lora_support": True
            }
        }
    )
    print("Deployment created")
    print("Deployment taks ~5 min...")

In [ ]:
# Deployment status
start = time()
while True:
    deployment = nemo.deployment.model_deployments.retrieve(deployment_name="nim", namespace=NS)
    status = deployment.status_details.status
    if status == 'ready':
        break
    elapsed = int(time() - start)
    print(f"\rStatus: {status} | Elapsed: {elapsed//60}m {elapsed%60}s", end="")
    sleep(10)

print("\nModel deployed")

## Step 5: Test Inference

Verify the deployed model responds to embedding requests.

In [ ]:
# 5. Test inference
client = OpenAI(base_url=f"{NIM_URL}/v1", api_key="None")
emb = client.embeddings.create(
    input=["Deep learning for computer vision"], 
    model=f"{NS}/model", 
    extra_body={"input_type": "query"})

print(f"Inference works! Embedding dimension: {len(emb.data[0].embedding)}")


## Step 6: Evaluate Performance

Run the SciDocs benchmark to measure retrieval quality (~10 minutes).

In [ ]:
# 6. Evaluate on SciDocs https://huggingface.co/datasets/BeIR/scidocs (~10 min)
# Evaluator: Measures model performance on benchmarks https://docs.nvidia.com/nim-operator/latest/evaluator.html
print("Creating evaluation config...")

eval_cfg = {
    "type": "retriever",
    "namespace": NS,
    "tasks": {
        "scidocs": {
            "type": "beir",
            "dataset": {"files_url": "file://scidocs/"},
            "metrics": {"recall_5": {"type": "recall_5"}}}}}

target = {
    "type": "retriever",
    "retriever": {
        "pipeline": {
            "query_embedding_model": {
                "api_endpoint": {"url": f"{NIM_URL}/v1/embeddings", "model_id": f"{NS}/model"}},
            "index_embedding_model": {
                "api_endpoint": {"url": f"{NIM_URL}/v1/embeddings", "model_id": f"{NS}/model"}},
            "top_k": 10}}}

print("Running evaluation...")
ejob = nemo.evaluation.jobs.create(config=eval_cfg, target=target)
print(f"Evaluation job id: {ejob.id}")
print("Evaluation takes ~10 minutes...")

In [ ]:
# Evaluation status
start = time()
while True:
    eval_status = nemo.evaluation.jobs.retrieve(ejob.id)
    if eval_status.status not in ["pending", "created", "running"]:
        break
    elapsed = int(time() - start)
    print(f"\r{elapsed//60}m {elapsed%60}s", end="")
    sleep(30)

print(f"\nEvaluation complete | Time: {int(time() - start)//60}m")
res = nemo.evaluation.jobs.results(ejob.id)

## Step 7: Display Results

View the evalaution results and measure improvment over baseline.

In [ ]:
# 7. Display results
baseline = 0.159
yours = res.tasks['scidocs'].metrics['retriever.recall_5'].scores['recall_5'].value
improvement = ((yours / baseline) - 1) * 100

print("="*70)
print("EVALUATION RESULTS - SciDocs Benchmark")
print("="*70)
print(f"Baseline (pretrained): {baseline:.3f}")
print(f"Fine-tuned model:      {yours:.3f}")
print(f"Improvement:          +{improvement:.1f}%")
print("="*70)
print(f"\nModel deployed at: {NIM_URL}/v1/embeddings")
print(f"Model name: {NS}/model")

## Summary

**What You Built:**
- Fine-tuned embedding model optimized for scientific paper retrieval
- Production NIM deployment serving embeddings via OpenAI-compatible API
- Measured 6-10% improvement in recall@5 on SciDocs benchmark

**Key Results:**
- Baseline: 0.159 recall@5
- **Impact:** Your model finds relevant papers in top-5 results 6-10% more often than baseline


## Next Steps

**Scale Up:**
- Train on full SPECTER dataset for additional improvement
- Increase to 3 epochs for better convergence

**Apply to Your Domain:**
- [Format your data as query-positive-negative triplets](https://docs.nvidia.com/nemo/microservices/latest/fine-tune/models/embedding.html#data-preparation)
- Replace SPECTER dataset with your domain data (legal, medical, product catalogs, etc.)
- Evaluate on your own retrieval tasks

**Learn More:**
- [NeMo Microservices Documentation](https://docs.nvidia.com/nemo/microservices/latest/)
- [Embedding Model Guide](https://build.nvidia.com/nvidia/llama-3_2-nv-embedqa-1b-v2)
- [Other NeMo Tutorials](../../../README.md)


## Cleanup (Optional)

Run cells below to delete resources. Execute only what you need to clean up.

In [ ]:
# Delete deployment (running NIM service - frees GPU, keeps model for redeployment)
print("Deleting deployment...")
nemo.deployment.model_deployments.delete(deployment_name="nim", namespace=NS)
print("Deployment deleted")

In [ ]:
# Delete model artifacts (fine-tuned weights - frees storage, can't redeploy)
print("Deleting models...")
for m in nemo.models.list(filter={"namespace": NS}).data:
    nemo.models.delete(namespace=NS, model_name=m.name.split('/')[-1])
print("Models deleted")

In [ ]:
# Delete training data (dataset, repo, and local files)
print("Deleting dataset...")
nemo.datasets.delete(namespace=NS, dataset_name="data")
print("Deleting data repo...")
hf.delete_repo(f"{NS}/data", repo_type='dataset')
print("Dataset deleted")

In [ ]:
# Delete configs (training/evaluation templates)
print("Deleting configs...")
nemo.customization.configs.delete(config_name="cfg@v1", namespace=NS)
for cfg in nemo.evaluation.configs.list(filter={"namespace": NS}).data:
    nemo.evaluation.configs.delete(config_id=cfg.id)
print("Configs deleted")

In [ ]:
# Delete eval jobs (evaluation run history)
print("Deleting eval jobs...")
for job in nemo.evaluation.jobs.list(filter={"namespace": NS}).data:
    nemo.evaluation.jobs.delete(job_id=job.id)
print("Eval jobs deleted")

In [ ]:
# Delete namespace
print("Deleting namespace...")
nemo.namespaces.delete(id=NS)
requests.delete(f"{NDS_URL}/v1/datastore/namespaces/{NS}")
print(f"Namespace {NS} deleted")